# Review Sentiment

- Select `METHOD = "rule"` or `METHOD = "model"` to compute review-level sentiment and write it to `review_sentiment`.
- Set `OVERWRITE = True` to recompute all reviews, or `False` to only insert rows for reviews without sentiment.

In [4]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

DB_PATH = Path("../data/app.sqlite")
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

In [16]:
RULE_VERSION = "v1"

# METHOD = "rule"
METHOD = "model"

OVERWRITE = False

In [6]:
import re

LEXICON = {
    "good": 1.0,
    "great": 1.5,
    "amazing": 2.0,
    "excellent": 2.0,
    "love": 1.5,
    "like": 1.0,
    "bad": -1.0,
    "terrible": -2.0,
    "awful": -2.0,
    "hate": -1.5,
    "dislike": -1.0,
    "poor": -1.0,
    "worst": -2.0,
    "좋다": 1.0,
    "최고": 2.0,
    "훌륭": 1.5,
    "맛있": 1.5,
    "추천": 1.0,
    "만족": 1.0,
    "친절": 1.0,
    "별로": -1.0,
    "나쁘": -1.0,
    "최악": -2.0,
    "불친절": -1.5,
    "실망": -1.0,
}


def _tokenize(text: str) -> list[str]:
    return re.findall(r"[A-Za-z']+|[가-힣]+", (text or "").lower())


def compute_rule_sentiment(text: str) -> tuple[float, float, str, str]:
    tokens = _tokenize(text)
    if not tokens:
        return 0.0, 0.0, "rule", RULE_VERSION
    score = 0.0
    matched = 0
    for token in tokens:
        weight = LEXICON.get(token)
        if weight is None:
            continue
        matched += 1
        score += weight
    confidence = matched / len(tokens)
    return score, confidence, "rule", RULE_VERSION


def load_model_pipeline():
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        use_safetensors=True,
    )
    return pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        max_length=256,
        batch_size=16,
    )


def compute_model_sentiment(text: str, pipe) -> tuple[float, float, str, str]:
    result = pipe(text, truncation=True)[0]
    label = str(result.get("label", "")).lower()
    score = float(result.get("score", 0.0))
    if "negative" in label:
        polarity = -score
    elif "positive" in label:
        polarity = score
    else:
        polarity = 0.0
    return polarity, score, "model", MODEL_NAME

In [7]:
conn = sqlite3.connect(DB_PATH)

In [14]:
pd.read_sql(
        "SELECT * FROM review_sentiment",
        conn,
    )

,review_id,sentiment_raw,confidence,method,version,created_at


In [13]:
conn.executescript(
#        """ DELETE FROM sqlite_sequence WHERE name = 'review_sentiment'; """
#        """ DELETE FROM review_sentiment; """
)
conn.commit()

In [17]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"DB not found at {DB_PATH.resolve()}")
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
def ensure_review_sentiment(conn):
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS review_sentiment (
          review_id INTEGER NOT NULL,
          sentiment_raw REAL NOT NULL,
          confidence REAL NOT NULL,
          method TEXT NOT NULL,
          version TEXT,
          created_at TEXT,
          PRIMARY KEY (review_id, method, version),
          FOREIGN KEY (review_id) REFERENCES review(id)
        );
        """
    )
ensure_review_sentiment(conn)
reviews = pd.read_sql(
    "SELECT id AS review_id, content FROM review",
    conn,
)


In [18]:
print(f"Reviews queued for sentiment: {len(reviews)}")
if len(reviews) > 0:
    display(reviews.head(10))

Reviews queued for sentiment: 43


,review_id,content
0,1,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...
1,2,처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정...
2,3,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...
3,4,비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...
4,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...
5,6,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...
6,7,교촌치킨에서 간장 레드 반반 주문. 간장만 먹으면 심심하고 레드만 먹으면 자극적이면...
7,8,땡겨요에서 할인을 크게 해서 한동안 집앞 자담치킨 참 많이도 포장해 먹었다. 왓더핫...
8,9,굽네치킨에서 유일하게 좋아하는 고추바사삭. 근데 살이 너무 없는것 같다
9,10,성남 현대시장 명성왕족발에서 족발 대자 포장. 싸고 맛 좋고 특히 온누리상품권을 쓸...


In [19]:
if reviews.empty:
    raise ValueError("review is empty; nothing to compute.")
if not OVERWRITE:
    existing = pd.read_sql(
        "SELECT DISTINCT review_id FROM review_sentiment",
        conn,
    )
    if not existing.empty:
        reviews = reviews[~reviews["review_id"].isin(existing["review_id"])]
if reviews.empty:
    conn.close()
    print("No reviews to compute sentiment for.")
else:
    print(f"Reviews to compute sentiment: {len(reviews)}")
    display(reviews.head(10))

Reviews to compute sentiment: 43


,review_id,content
0,1,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...
1,2,처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정...
2,3,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...
3,4,비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...
4,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...
5,6,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...
6,7,교촌치킨에서 간장 레드 반반 주문. 간장만 먹으면 심심하고 레드만 먹으면 자극적이면...
7,8,땡겨요에서 할인을 크게 해서 한동안 집앞 자담치킨 참 많이도 포장해 먹었다. 왓더핫...
8,9,굽네치킨에서 유일하게 좋아하는 고추바사삭. 근데 살이 너무 없는것 같다
9,10,성남 현대시장 명성왕족발에서 족발 대자 포장. 싸고 맛 좋고 특히 온누리상품권을 쓸...


In [20]:
if not reviews.empty:
    if METHOD == "model":
        pipe = load_model_pipeline()
        sentiment = reviews["content"].apply(lambda text: compute_model_sentiment(text, pipe))
    elif METHOD == "rule":
        sentiment = reviews["content"].apply(compute_rule_sentiment)
    else:
        raise ValueError(f"Unknown METHOD: {METHOD}")
    reviews["sentiment_raw"] = sentiment.map(lambda item: item[0])
    reviews["confidence"] = sentiment.map(lambda item: item[1])
    reviews["method"] = sentiment.map(lambda item: item[2])
    reviews["version"] = sentiment.map(lambda item: item[3])
    reviews["created_at"] = datetime.now(timezone.utc).isoformat()
    payload = list(
        reviews[
            [
                "review_id",
                "sentiment_raw",
                "confidence",
                "method",
                "version",
                "created_at",
            ]
        ].itertuples(index=False, name=None)
    )
    conn.executemany(
        """
        INSERT OR REPLACE INTO review_sentiment
        (review_id, sentiment_raw, confidence, method, version, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        payload,
    )
    conn.commit()
    conn.close()
    print(f"Inserted/updated {len(payload)} sentiment rows using METHOD={METHOD}.")

Device set to use cpu


Inserted/updated 43 sentiment rows using METHOD=model.


In [21]:
conn = sqlite3.connect(DB_PATH)

pd.read_sql(
        "SELECT * FROM review_sentiment",
        conn,
    )

,review_id,sentiment_raw,confidence,method,version,created_at
0,1,0.530003,0.530003,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
1,2,0.000000,0.379968,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
2,3,-0.635604,0.635604,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
3,4,0.000000,0.572736,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
4,5,0.694385,0.694385,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
5,6,0.643638,0.643638,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
6,7,0.634538,0.634538,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
7,8,0.000000,0.360365,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
8,9,-0.496433,0.496433,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00
9,10,0.814621,0.814621,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,2026-01-14T06:12:10.531865+00:00


# Review-Node Relevance

Compute max-similarity between each review and its assigned node using sentence embeddings.


In [1]:
import sqlite3
import re
import pandas as pd

from sentence_transformers import SentenceTransformer, util
from pathlib import Path

DB_PATH = Path("../data/app.sqlite")
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

In [2]:
# weights (조정 가능)
W_NAME = 1.0
W_TAXONOMY = 0.5
W_RELATION = 0.7

In [3]:
conn = sqlite3.connect(DB_PATH)

# 1. base review-node (review unit, no node_name duplication logic here)
query = """
SELECT r.id AS review_id,
       r.content AS review_content,
       r.node_id
FROM review r
WHERE r.node_id IS NOT NULL
"""
review_nodes = pd.read_sql(query, conn)

# node names: node unit (1 row per node)
node_df = pd.read_sql(
    """
    SELECT id AS node_id,
           name AS node_name
    FROM nodes
    """,
    conn,
)

# 2. taxonomy (already unique by schema; do NOT normalize/dedup text)
taxonomy_df = pd.read_sql(
    """
    SELECT nt.node_id,
           t.key || ':' || t.value AS text
    FROM node_taxonomy nt
    JOIN taxonomy t ON t.id = nt.taxonomy_id
    """,
    conn,
)

# 3. relations (already unique by schema)
relation_df = pd.read_sql(
    """
    SELECT e.child_id AS node_id,
           e.relation || ':' || p.name AS text
    FROM edges e
    JOIN nodes p ON p.id = e.parent_id
    """,
    conn,
)

conn.close()

## Basic

In [4]:
# 4. build structured repr map
# - name: node table 기준 (1회)
# - taxonomy / relation: schema 그대로 수집
# - 동일 문자열이 여러 경로에서 들어오면 그대로 유지 (에러 탐지 목적)
repr_map: dict[int, dict[str, list[str]]] = {}

def ensure_node(node_id: int):
    repr_map.setdefault(
        node_id,
        {"name": [], "taxonomy": [], "relation": []},
    )

# name: node 단위 (중복 제거는 여기서만)
for _, r in node_df.iterrows():
    ensure_node(r.node_id)
    if r.node_name:
        repr_map[r.node_id]["name"].append(r.node_name)

# taxonomy: 그대로 (category / Category 모두 유지)
for _, r in taxonomy_df.iterrows():
    ensure_node(r.node_id)
    if r.text:
        repr_map[r.node_id]["taxonomy"].append(r.text)

# relation: 그대로
for _, r in relation_df.iterrows():
    ensure_node(r.node_id)
    if r.text:
        repr_map[r.node_id]["relation"].append(r.text)

# materialize node_repr (표시/검증용)
def build_node_repr(node_id: int) -> str:
    parts = repr_map.get(node_id, {})
    texts: list[str] = []
    for k in ("name", "taxonomy", "relation"):
        texts.extend(parts.get(k, []))
    return " | ".join(texts)

review_nodes["node_repr"] = review_nodes["node_id"].apply(build_node_repr)

# node_name join (표시용)
review_nodes = review_nodes.merge(
    node_df, on="node_id", how="left"
)

# 5. utils
def split_sentences(text: str) -> list[str]:
    try:
        parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
        return [p for p in parts if p] or [text or ""]
    except Exception:
        return [text or ""]

def max_sim(review_emb, texts: list[str], model) -> float:
    if not texts:
        return 0.0
    emb = model.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(review_emb, emb).max().item())

# 6. similarity
if review_nodes.empty:
    relevance_df = pd.DataFrame(
        columns=[
            "review_id",
            "review_content",
            "node_id",
            "node_name",
            "node_repr",
            "score_name",
            "score_taxonomy",
            "score_relation",
            "score_total",
        ]
    )
else:
    model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)

    def compute_scores(row):
        review_sents = split_sentences(row.review_content)
        if not review_sents:
            return pd.Series([0.0, 0.0, 0.0, 0.0])

        review_emb = model.encode(
            review_sents, convert_to_tensor=True, normalize_embeddings=True
        )

        parts = repr_map.get(row.node_id, {})

        s_name = max_sim(review_emb, parts.get("name", []), model)
        s_tax = max_sim(review_emb, parts.get("taxonomy", []), model)
        s_rel = max_sim(review_emb, parts.get("relation", []), model)

        s_total = (
            W_NAME * s_name
            + W_TAXONOMY * s_tax
            + W_RELATION * s_rel
        )

        return pd.Series([s_name, s_tax, s_rel, s_total])

    review_nodes[
        ["score_name", "score_taxonomy", "score_relation", "score_total"]
    ] = review_nodes.apply(compute_scores, axis=1)

    relevance_df = (
        review_nodes[
            [
                "review_id",
                "review_content",
                "node_id",
                "node_name",
                "node_repr",
                "score_name",
                "score_taxonomy",
                "score_relation",
                "score_total",
            ]
        ]
        .sort_values("score_total", ascending=False)
        .reset_index(drop=True)
    )

In [5]:
relevance_df.sort_values('score_total')

,review_id,review_content,node_id,node_name,node_repr,score_name,score_taxonomy,score_relation,score_total
46,43,저번에 이마트에서 6개 만원 세일했는데 요샌 기린 맥주와 함께 만원 5개 세일중,87,산토리 나마비루,산토리 나마비루 | category:beverage,0.361596,0.140542,0.000000,0.431867
45,11,귀한족발에서 족발 배달시켰는데 막국수 반찬까지 있는건 좋지만 정작 족발맛이 아쉽다,23,귀한족발,귀한족발 | domain:food,0.395515,0.075064,0.000000,0.433047
44,6,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...,57,푸라닭,푸라닭 | domain:food,0.440629,0.088911,0.000000,0.485085
43,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,6,노랑통닭,노랑통닭 | domain:food,0.450659,0.071367,0.000000,0.486342
42,19,정자동 정동마트에서 종종 바나나 세일을 하는데 대부분 거무죽죽하니 신선도 관리를 어...,34,정동마트 (정자동),정동마트 (정자동) | category:store,0.442277,0.094751,0.000000,0.489653
41,13,마트에서 풀무원 감자전/오징어부추전 구입. 에어프라이로 조리했는데 기름 범벅. 후라...,89,풀무원,풀무원 | domain:food,0.477579,0.071015,0.000000,0.513087
40,33,고기 두툼하고 잡내 없어 맛있다. 고기가 덜익은 건 아닌데 조리법보다 좀더 익혀도 ...,90,CJ고메 대가 탕수육,CJ고메 대가 탕수육 | category:food,0.496025,0.086313,0.000000,0.539182
39,26,윙봉으로만 구성되 먹기는 편한데 닭에서 냄새가 좀 나는 것 같다. 국산일까? 수입산...,52,바사칸 윙 (BBQ),바사칸 윙 (BBQ) | category:fried chicken,0.488551,0.178478,0.000000,0.577790
38,4,비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...,10,맵소디 (BBQ),맵소디 (BBQ) | category:fried chicken,0.532633,0.111694,0.000000,0.588479
37,21,4개 한셋트 치약 1+1 세일한다기에 사고 보니 3개 한셋트 치약이었다. 영수증엔 ...,36,이마트,이마트 | domain:retail,0.543285,0.105699,0.000000,0.596135


## Calibration

In [4]:
# ---------- scoring controls ----------

# (1) soft aggregation
TOP_K = 3                   # 0 or None → use plain max
MAX_THRESHOLD = 0.35        # 0 or None → no gating

# (2) node_name penalty
NAME_PENALTY_FACTOR = 0.3   # 0 or None → no penalty
NAME_MIN_FOR_PENALTY = 0.25 # 이 값 이상이면 패널티 적용 안 함

# (3) taxonomy exclusion signal
TAX_LOW_THRESHOLD = 0.15    # 0 or None → no exclusion
TAX_PENALTY_FACTOR = 0.6
MIN_TOTAL_FOR_TAX_PENALTY = 0.2  # 이 값 이하이면 taxonomy로 더 깎지 않음


In [5]:
def soft_gate(s: float, t: float) -> float:
    """
    threshold 이하를 0으로 자르지 않고 연속적으로 감쇠
    """
    if not t or t <= 0:
        return s
    if s <= t:
        return s * (s / t)   # 0으로 붕괴 방지
    return s


def soft_max_sim(review_emb, texts: list[str], model) -> float:
    if not texts:
        return 0.0

    emb = model.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(review_emb, emb).flatten()

    # top-k 평균 or max
    if TOP_K and TOP_K > 0:
        k = min(TOP_K, len(scores))
        s = float(scores.topk(k).values.mean())
    else:
        s = float(scores.max())

    # soft gate 적용
    return soft_gate(s, MAX_THRESHOLD)


In [6]:
def has_node_token(review_text: str, node_name: str) -> bool:
    if not node_name:
        return False
    tokens = node_name.split()
    return any(t in review_text for t in tokens)



def compute_scores(row):
    review_sents = split_sentences(row.review_content)
    if not review_sents:
        return pd.Series([0.0, 0.0, 0.0, 0.0])

    review_emb = model.encode(
        review_sents, convert_to_tensor=True, normalize_embeddings=True
    )

    parts = repr_map.get(row.node_id, {})

    # (1) soft aggregation
    s_name = soft_max_sim(review_emb, parts.get("name", []), model)
    s_tax  = soft_max_sim(review_emb, parts.get("taxonomy", []), model)
    s_rel  = soft_max_sim(review_emb, parts.get("relation", []), model)

    # (2) node_name penalty (이미 어느 정도 맞으면 패널티 미적용)
    if (
        NAME_PENALTY_FACTOR
        and NAME_PENALTY_FACTOR > 0
        and s_name < NAME_MIN_FOR_PENALTY
        and not has_node_token(row.review_content, row.node_name)
    ):
        s_name *= NAME_PENALTY_FACTOR

    # base total
    s_total = (
        W_NAME * s_name
        + W_TAXONOMY * s_tax
        + W_RELATION * s_rel
    )

    # (3) taxonomy exclusion (총점이 이미 낮으면 더 깎지 않음)
    if (
        TAX_LOW_THRESHOLD
        and TAX_LOW_THRESHOLD > 0
        and s_tax < TAX_LOW_THRESHOLD
        and s_total > MIN_TOTAL_FOR_TAX_PENALTY
        and TAX_PENALTY_FACTOR
        and TAX_PENALTY_FACTOR > 0
    ):
        s_total *= TAX_PENALTY_FACTOR

    return pd.Series([s_name, s_tax, s_rel, s_total])


In [7]:
# 4. build structured repr map
# - name: node table 기준 (1회)
# - taxonomy / relation: schema 그대로 수집
# - 동일 문자열이 여러 경로에서 들어오면 그대로 유지 (에러 탐지 목적)
repr_map: dict[int, dict[str, list[str]]] = {}

def ensure_node(node_id: int):
    repr_map.setdefault(
        node_id,
        {"name": [], "taxonomy": [], "relation": []},
    )

# name: node 단위 (중복 제거는 여기서만)
for _, r in node_df.iterrows():
    ensure_node(r.node_id)
    if r.node_name:
        repr_map[r.node_id]["name"].append(r.node_name)

# taxonomy: 그대로 (category / Category 모두 유지)
for _, r in taxonomy_df.iterrows():
    ensure_node(r.node_id)
    if r.text:
        repr_map[r.node_id]["taxonomy"].append(r.text)

# relation: 그대로
for _, r in relation_df.iterrows():
    ensure_node(r.node_id)
    if r.text:
        repr_map[r.node_id]["relation"].append(r.text)

# materialize node_repr (표시/검증용)
def build_node_repr(node_id: int) -> str:
    parts = repr_map.get(node_id, {})
    texts: list[str] = []
    for k in ("name", "taxonomy", "relation"):
        texts.extend(parts.get(k, []))
    return " | ".join(texts)

review_nodes["node_repr"] = review_nodes["node_id"].apply(build_node_repr)

# node_name join (표시용)
review_nodes = review_nodes.merge(
    node_df, on="node_id", how="left"
)

# 5. utils
def split_sentences(text: str) -> list[str]:
    try:
        parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
        return [p for p in parts if p] or [text or ""]
    except Exception:
        return [text or ""]

def max_sim(review_emb, texts: list[str], model) -> float:
    if not texts:
        return 0.0
    emb = model.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(review_emb, emb).max().item())

# 6. similarity
if review_nodes.empty:
    relevance_df = pd.DataFrame(
        columns=[
            "review_id",
            "review_content",
            "node_id",
            "node_name",
            "node_repr",
            "score_name",
            "score_taxonomy",
            "score_relation",
            "score_total",
        ]
    )
else:
    model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)

    review_nodes[
        ["score_name", "score_taxonomy", "score_relation", "score_total"]
    ] = review_nodes.apply(compute_scores, axis=1)

    relevance_df = (
        review_nodes[
            [
                "review_id",
                "review_content",
                "node_id",
                "node_name",
                "node_repr",
                "score_name",
                "score_taxonomy",
                "score_relation",
                "score_total",
            ]
        ]
        .sort_values("score_total", ascending=False)
        .reset_index(drop=True)
    )

In [8]:
relevance_df.sort_values('score_total')

,review_id,review_content,node_id,node_name,node_repr,score_name,score_taxonomy,score_relation,score_total
46,34,바삭한 빵? 과자 속 크림이 듬뿍. 느끼하지 않아 좋다.,64,피코크 커스터드쿠키슈,피코크 커스터드쿠키슈 | category:bakery | produces:피코크,0.052765,0.063822,0.327151,0.188209
45,13,마트에서 풀무원 감자전/오징어부추전 구입. 에어프라이로 조리했는데 기름 범벅. 후라...,89,풀무원,풀무원 | domain:food,0.373888,0.002649,0.000000,0.225127
44,43,저번에 이마트에서 6개 만원 세일했는데 요샌 기린 맥주와 함께 만원 5개 세일중,87,산토리 나마비루,산토리 나마비루 | category:beverage,0.361596,0.056434,0.000000,0.233888
43,11,귀한족발에서 족발 배달시켰는데 막국수 반찬까지 있는건 좋지만 정작 족발맛이 아쉽다,23,귀한족발,귀한족발 | domain:food,0.395515,0.016099,0.000000,0.242138
42,5,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,6,노랑통닭,노랑통닭 | domain:food,0.407170,0.007767,0.000000,0.246632
41,21,4개 한셋트 치약 1+1 세일한다기에 사고 보니 3개 한셋트 치약이었다. 영수증엔 ...,36,이마트,이마트 | domain:retail,0.409354,0.004571,0.000000,0.246984
40,6,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...,57,푸라닭,푸라닭 | domain:food,0.426726,0.013718,0.000000,0.260151
39,26,윙봉으로만 구성되 먹기는 편한데 닭에서 냄새가 좀 나는 것 같다. 국산일까? 수입산...,52,바사칸 윙 (BBQ),바사칸 윙 (BBQ) | category:fried chicken,0.411569,0.050660,0.000000,0.262139
38,19,정자동 정동마트에서 종종 바나나 세일을 하는데 대부분 거무죽죽하니 신선도 관리를 어...,34,정동마트 (정자동),정동마트 (정자동) | category:store,0.442277,0.025651,0.000000,0.273062
37,18,예전에 땡겨요에서 세일하길래 무슨 요리 배달 주문했더니 배달에 한시간 이상 걸린다고...,14,땡겨요,땡겨요 | category:delivery | mode:online,0.450762,0.026887,0.000000,0.278523
